## Score - 0.84463
- #### updated feature engineering by removing unnecessary features 

In [161]:
import numpy as np 
import pandas as pd  
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [162]:
pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/SampleSubmission.csv').sample(5)

,ID,TargetF1,TargetRAUC
87,ID_84316E57,0,0
116,ID_AF250671,0,0
992,ID_17D6C3FF,0,0
960,ID_AE1D5DDC,0,0
909,ID_36C7DD36,0,0


In [163]:
data_dict =pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/data_dictionary.csv')

In [164]:
df = pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/Train.csv')

In [165]:
df["is_climate_sensitive"].value_counts()

is_climate_sensitive
1    2047
0    1099
Name: count, dtype: int64

In [166]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3146 entries, 0 to 3145
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID                    3146 non-null   object 
 1   zone                  3146 non-null   object 
 2   gender                3146 non-null   object 
 3   deathdate             3146 non-null   object 
 4   age                   3146 non-null   float64
 5   avg_temperature       3146 non-null   float64
 6   max_temperature       3146 non-null   float64
 7   min_temperature       3146 non-null   float64
 8   precipitation         3146 non-null   float64
 9   latitude              3146 non-null   float64
 10  longitude             3146 non-null   float64
 11  location              3146 non-null   object 
 12  is_climate_sensitive  3146 non-null   int64  
dtypes: float64(7), int64(1), object(5)
memory usage: 319.6+ KB


In [167]:
def preprocessor(
    data,
    fit=True,
    target=None,
    te_maps=None,
    maxloc=1
):
    data = data.copy()

    data["zone"] = (data["zone"] == "Peri_urban").astype(int)
    data["gender"] = (data["gender"] == "Male").astype(int)

    data["age"] = data["age"].astype(int)

    data["age_group"] = pd.cut(
        data["age"],
        bins=[-1, 10, 20, 30, 40, 50, 60, 70, 80, 90, np.inf],
        labels=False
    ) + 1

    data["is_young"] = (data["age"] <= 20).astype(int)

    data["is_rain"] = (data["precipitation"] > 0).astype(int)


    data["precip_log"] = np.log1p(data["precipitation"])

    data["precip_per_temp"] = (
        data["precipitation"] /
        (abs(data["avg_temperature"]) + 1)
    )

    data["temp_rain_interaction"] = (
        data["avg_temperature"] *
        data["precipitation"]
    )

    data["temp_range"] = (
        data["max_temperature"] -
        data["min_temperature"]
    )

    data["avg_temp_position"] = (
        (data["avg_temperature"] - data["min_temperature"]) /
        (data["max_temperature"] - data["min_temperature"] + 1e-6)
    )

    loc_cols = [f"location_{i+1}" for i in range(maxloc)]

    location_split = data["location"].str.split(",", expand=True)

    for i, col in enumerate(loc_cols):
        if i < location_split.shape[1]:
            data[col] = location_split[i].str.strip()
        else:
            data[col] = "Unknown"

    data["deathdate"] = pd.to_datetime(
        data["deathdate"],
        errors="coerce"
    )

    data["date"] = data["deathdate"].dt.day
    data["month"] = data["deathdate"].dt.month
    data["year"] = data["deathdate"].dt.year
    data["day_of_week"] = data["deathdate"].dt.dayofweek
    data["day_of_year"] = data["deathdate"].dt.dayofyear
    data["week_of_year"] = (
        data["deathdate"].dt.isocalendar().week.astype(int)
    )

    te_cols = ["zone", "gender", "age_group"] + loc_cols

    if fit:
        if target is None:
            raise ValueError("target must be provided when fit=True")

        global_mean = data[target].mean()
        te_maps = {}

        for col in te_cols:
            stats = (
                pd.DataFrame({
                    "feature": data[col],
                    "target": data[target]
                })
                .groupby("feature")["target"]
                .agg(["mean", "count"])
            )

            smoothing = 10

            stats["encoded"] = (
                stats["count"] * stats["mean"]
                + smoothing * global_mean
            ) / (
                stats["count"] + smoothing
            )

            te_maps[col] = {
                "mapping": stats["encoded"].to_dict(),
                "global_mean": global_mean
            }

            data[f"{col}_te"] = (
                data[col]
                .map(te_maps[col]["mapping"])
                .fillna(global_mean)
            )

    else:
        if te_maps is None:
            raise ValueError("te_maps must be provided when fit=False")

        for col in te_cols:
            data[f"{col}_te"] = (
                data[col]
                .map(te_maps[col]["mapping"])
                .fillna(te_maps[col]["global_mean"])
            )

    for col in loc_cols:
        data[f"young_{col}"] = (
            data[f"{col}_te"] * data["is_young"]
        )


    data["zone_gender"] = (
        data["zone_te"] * data["gender_te"]
    )



    data["latitude_gender"] = (
        data["latitude"] * data["gender_te"]
    )

    data["longitude_gender"] = (
        data["longitude"] * data["gender_te"]
    )

    data["longitude_zone"] = (
        data["longitude"] * data["zone_te"]
    )


    data.drop(
        columns=["deathdate", "location","zone", "zone_te"]+loc_cols ,
        inplace=True
    )

    if fit:
        return data, te_maps

    return data

In [168]:
df1, enc_dict = preprocessor(df, fit=True, target='is_climate_sensitive')

In [169]:
X = df1.drop(columns=["is_climate_sensitive", "ID", ])
y = df1["is_climate_sensitive"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [170]:
models = {

    "Gradient Boosting": GradientBoostingClassifier(
        learning_rate=0.01,
        min_samples_leaf=10,
        min_samples_split=15,
        n_estimators=300,
        random_state=42,
        subsample=0.9,
        max_depth=3
    ),

    "Extra Trees": ExtraTreesClassifier(
        max_depth=5,
        max_features=None,
        min_samples_leaf=2,
        min_samples_split=5,
        n_estimators=500,
        n_jobs=-1,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        max_depth=5,
        max_features=None,
        min_samples_split=10,
        n_estimators=700,
        n_jobs=-1,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        colsample_bytree=1.0,
        eval_metric="logloss",
        gamma=0.1,
        learning_rate=0.01,
        max_depth=4,
        min_child_weight=3,
        n_estimators=300,
        n_jobs=-1,
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        learning_rate=0.01,
        n_estimators=300,

        max_depth=4,
        num_leaves=15,

        min_child_samples=10,
        min_split_gain=0.1,

        subsample=0.9,
        colsample_bytree=1.0,

        reg_alpha=0.0,
        reg_lambda=0.0,

        objective="binary",
        verbosity=-1,
        random_state=42,
        n_jobs=-1
    ),

    "CatBoost": CatBoostClassifier(
        learning_rate=0.01,
        iterations=300,

        depth=4,

        l2_leaf_reg=3,
        min_data_in_leaf=10,

        random_strength=1.0,
        rsm=1.0,

        loss_function="Logloss",
        eval_metric="AUC",

        random_seed=42,
        verbose=False,
        thread_count=-1
    )
}
tuned_models = {}
results = []

for name, model in models.items():

    model.fit(X_train, y_train)
    tuned_models[name] = model

    train_prob = model.predict_proba(X_train)[:, 1]
    train_pred = (train_prob >= 0.5).astype(int)

    train_f1 = f1_score(y_train, train_pred)
    train_auc = roc_auc_score(y_train, train_prob)
    train_score = 0.6 * train_f1 + 0.4 * train_auc

    test_prob = model.predict_proba(X_test)[:, 1]
    test_pred = (test_prob >= 0.5).astype(int)

    test_f1 = f1_score(y_test, test_pred)
    test_auc = roc_auc_score(y_test, test_prob)
    test_score = 0.6 * test_f1 + 0.4 * test_auc

    results.append({
        "Model": name,

        "Train F1": train_f1,
        "Train ROC-AUC": train_auc,
        "Train Final Score": train_score,

        "Test F1": test_f1,
        "Test ROC-AUC": test_auc,
        "Test Final Score": test_score,

        "Parameters": model.get_params()
    })

results_df = pd.DataFrame(results)

In [171]:
results_df = pd.DataFrame(results)

results_df["Train-Test Final Score Difference"] = (
    results_df["Train Final Score"] - results_df["Test Final Score"]
).abs()

results_df = results_df.sort_values(
    by=[
        "Train-Test Final Score Difference",
        "Test Final Score"
    ],
    ascending=[
        True,
        False
    ]
).reset_index(drop=True)

results_df

,Model,Train F1,Train ROC-AUC,Train Final Score,Test F1,Test ROC-AUC,Test Final Score,Parameters,Train-Test Final Score Difference
0,CatBoost,0.819798,0.844844,0.829817,0.826790,0.831585,0.828708,"{'iterations': 300, 'learning_rate': 0.01, 'de...",0.001108
1,Extra Trees,0.824409,0.862312,0.839570,0.821596,0.834545,0.826776,"{'bootstrap': False, 'ccp_alpha': 0.0, 'class_...",0.012795
2,Gradient Boosting,0.838842,0.864071,0.848934,0.826636,0.837561,0.831006,"{'ccp_alpha': 0.0, 'criterion': 'friedman_mse'...",0.017928
3,XGBoost,0.854858,0.884769,0.866822,0.822171,0.832162,0.826167,"{'objective': 'binary:logistic', 'base_score':...",0.040655
4,LightGBM,0.864417,0.891398,0.875209,0.821101,0.831608,0.825304,"{'boosting_type': 'gbdt', 'class_weight': None...",0.049906
5,Random Forest,0.864346,0.892531,0.875620,0.817757,0.831707,0.823337,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.052283


In [172]:
tf = pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/Test.csv')

In [173]:
tf1 = preprocessor(tf, fit=False, te_maps=enc_dict, maxloc=1)

In [174]:
tf1.head()

,ID,gender,age,avg_temperature,max_temperature,min_temperature,precipitation,latitude,longitude,age_group,...,day_of_year,week_of_year,gender_te,age_group_te,location_1_te,young_location_1,zone_gender,latitude_gender,longitude_gender,longitude_zone
0,ID_E760D84B,0,75,21.224387,24.632729,18.145633,4.295335,0.616667,33.500000,8,...,328,47,0.671414,0.297386,0.650668,0.000000,0.440893,0.414039,22.492372,21.998221
1,ID_6EDEA907,0,50,21.708596,25.184528,18.285274,1.376196,0.725422,33.098845,5,...,23,4,0.671414,0.497098,0.650668,0.000000,0.440893,0.487058,22.223031,21.734797
2,ID_B9FFC8D8,0,76,21.371149,24.584523,18.866490,2.336643,0.603194,33.542370,8,...,74,11,0.671414,0.297386,0.650668,0.000000,0.440893,0.404993,22.520819,22.026043
3,ID_74C6C94E,0,90,21.341990,25.188662,17.998317,4.703071,-0.590211,30.056939,9,...,107,16,0.671414,0.302667,0.650668,0.000000,0.430128,-0.396276,20.180652,19.255396
4,ID_0E02825D,1,8,19.710391,22.905167,18.082628,4.066863,0.603194,33.542370,1,...,112,17,0.631726,0.828350,0.650668,0.650668,0.414831,0.381053,21.189590,22.026043


In [175]:
df1.describe()

,gender,age,avg_temperature,max_temperature,min_temperature,precipitation,latitude,longitude,is_climate_sensitive,age_group,...,day_of_year,week_of_year,gender_te,age_group_te,location_1_te,young_location_1,zone_gender,latitude_gender,longitude_gender,longitude_zone
count,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,...,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000,3146.000000
mean,0.522886,24.243484,22.066541,26.206879,18.514193,1.519907,0.630242,33.407442,0.650668,3.111252,...,182.079148,26.541322,0.650662,0.656516,0.650268,0.399468,0.423373,0.410102,21.736643,21.737159
std,0.499555,31.428833,1.095122,1.691347,0.891524,1.621088,0.172728,0.579933,0.476835,2.923783,...,104.823557,14.988211,0.019826,0.212998,0.034749,0.318065,0.013848,0.113527,0.754285,0.441033
min,0.000000,0.000000,18.852803,20.884623,15.018778,0.000000,0.089819,30.782817,0.000000,1.000000,...,1.000000,1.000000,0.631726,0.297386,0.577437,0.000000,0.404703,0.056741,19.446308,20.213946
25%,0.000000,0.000000,21.319885,25.260527,17.954814,0.286814,0.564758,33.473515,0.000000,1.000000,...,95.000000,14.000000,0.631726,0.464661,0.632025,0.000000,0.414831,0.365934,21.151875,21.445554
50%,1.000000,3.000000,21.931299,26.081829,18.510478,1.034344,0.591961,33.483544,1.000000,1.000000,...,177.000000,26.000000,0.631726,0.828350,0.639921,0.632025,0.414831,0.384340,21.198698,21.978149
75%,1.000000,49.000000,22.652860,27.015877,19.040655,2.222345,0.608396,33.519695,1.000000,5.000000,...,271.000000,39.000000,0.671414,0.828350,0.672296,0.652445,0.440893,0.406286,22.480734,22.011154
max,1.000000,110.000000,27.012223,34.080947,21.849602,16.606523,1.187892,34.226647,1.000000,10.000000,...,366.000000,53.000000,0.671414,0.828350,0.786032,0.786032,0.440893,0.797567,22.980253,22.475383


In [176]:
def make_submission(
    test_data,
    model,
    id_col="ID"
):
    ids = test_data.pop(id_col)

    pred_binary = model.predict(test_data)
    pred_prob = model.predict_proba(test_data)[:, 1]
    submission = pd.DataFrame({
        "ID": ids,
        "TargetF1": pred_binary.astype(int),
        "TargetRAUC": pred_prob
    })

    return submission

In [178]:
best_name = results_df.iloc[0]["Model"]
best_model = tuned_models[best_name]

X_full = df1.drop(columns=["is_climate_sensitive", "ID"])
y_full = df1["is_climate_sensitive"]

best_model.fit(X_full, y_full)

X_submission = tf1.drop(columns=["ID"])
ids = tf1["ID"].copy()

pred_prob = best_model.predict_proba(X_submission)[:, 1]
pred_binary = (pred_prob >= 0.5).astype(int)

submission = pd.DataFrame({
    "ID": ids,
    "TargetF1": pred_binary,
    "TargetRAUC": pred_prob
})

submission.to_csv("climate-risk-fe-updated.csv", index=False)

print("Best model:", best_name)
print(submission.head())

Best model: CatBoost
            ID  TargetF1  TargetRAUC
0  ID_E760D84B         0    0.402286
1  ID_6EDEA907         1    0.596126
2  ID_B9FFC8D8         0    0.393152
3  ID_74C6C94E         0    0.402618
4  ID_0E02825D         1    0.863716
